In [ ]:
import os
import warnings
from optuna.storages import RetryFailedTrialCallback
import json
import optuna

experiment_name = input("Enter experiment name:")

print("Creating trials...")

def objective(trial: optuna.Trial):
    best_epoch = 0
    max_f1 = 0

    num_layers = trial.suggest_int('depth', 3, 7)
    filters = trial.suggest_int(f'filters', 16, 256) # values ignored
    kernel_size = trial.suggest_int(f'kernel', 3, 7)  # values ignored
    dropout_rate = trial.suggest_float("dropout", 0.2, 0.7)  # values ignored
    num_dense = trial.suggest_int('num_dense', 1, 3)  # values ignored
    dense_output = trial.suggest_int(f'dense', 16, 256)  # values ignored
    focal = trial.suggest_categorical(f'focal', [True, False])  # values ignored
    focal_factor = trial.suggest_int(f'focal_factor', 1, 10)  # values ignored

    run_id = trial.number + 27 if focal else trial.number
    
    os.makedirs(f"paper3_models_step2/{experiment_name}/{run_id}")

    with open(f'paper3_models_step2/{experiment_name}/{run_id}/model.json', 'w') as fp:
      json.dump({
        "depth": num_layers,
        "focal": focal,
        "focal_factor": focal_factor,
        "filters": filters,
        "kernel": kernel_size,
        "dropout": dropout_rate,
        "dense_depth": num_dense,
        "dense": dense_output
      }, fp)

    run_id += 1
    
    return max_f1

for focal in [False, True]:
    study = optuna.create_study(
      sampler=optuna.samplers.GridSampler({
          "depth": [3, 4, 5],
          "focal": [focal],
          "focal_factor": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10] if focal else [0],
          "filters": [64, 128, 256],
          "kernel": [5, 6, 7],
          "dropout": [0.7], #most common in top 10
          "num_dense": [3], #most common in top 10
          "dense": [128], #most common in top 10
      }),
      # storage = optuna.storages.JournalStorage(
      #     optuna.storages.journal.JournalFileBackend(file_path),
      # ),
      direction = "maximize",
      study_name=experiment_name + "_focal_" + str(focal),
      load_if_exists=True
    )
    
    study.optimize(objective, n_trials=3 * 10 * 3 * 3 if focal else 3 * 3 * 3)


